In [634]:
from dataclasses import dataclass, field

@dataclass(frozen=True)
class Grid:
    rows: int
    cols: int
    terminals: dict
    walls: frozenset = field(default_factory=frozenset)
    actions = dict(up=(-1, 0), right=(0, 1), down=(1, 0), left=(0, -1))
    arrows = {'up': "↑", 'right': "→", 'down': "↓", 'left': "←"}
    step_reward = 0
    def cell_repr(self, r, c):
        cell = (r, c)
        if cell in self.terminals:
            return self.terminals[cell]
        elif cell in self.walls:
            return '#'
        else:
            return '·'
    def render(self):
        for r in range(self.rows):
            for c in range(self.cols):
                print(f'{self.cell_repr(r, c):>3} ', end="")
            print()

    def step(self, cell, action):
        movement = self.actions[action]
        next_cell = (cell[0] + movement[0], cell[1] + movement[1])
        outside = not (0 <= next_cell[0] < self.rows and 0 <= next_cell[1] < self.cols)
        on_wall = next_cell in self.walls
        if outside or on_wall:
            next_cell = cell
            


        if next_cell in self.terminals:
            reward = self.terminals[next_cell]
        else:
            reward = self.step_reward
        return next_cell, reward


default_grid = Grid(3, 4, {(0, 3):1}, frozenset({(1, 1)}))
default_grid

Grid(rows=3, cols=4, terminals={(0, 3): 1}, walls=frozenset({(1, 1)}))

In [635]:
default_grid.render()

  ·   ·   ·   1 
  ·   #   ·   · 
  ·   ·   ·   · 


In [636]:
Grid.actions

{'up': (-1, 0), 'right': (0, 1), 'down': (1, 0), 'left': (0, -1)}

In [637]:
default_grid.actions

{'up': (-1, 0), 'right': (0, 1), 'down': (1, 0), 'left': (0, -1)}

In [638]:
cell = (0, 0)
for _ in range(3):
    cell, reward = default_grid.step(cell, 'right')
    print((cell, reward))


((0, 1), 0)
((0, 2), 0)
((0, 3), 1)


In [639]:
def show_V(grid: Grid, V):
    for r in range(grid.rows):
        for c in range(grid.cols):
            print(f'{round(V[r][c], 2):>4} ', end="")
        print()
    print('------------------')
def value_iteration(grid: Grid, gamma = 0.9):
    V = [[0 for _ in range(grid.cols)] for _ in range(grid.rows)]
    delta = 1e9
    while delta > 0.001:
        delta = 0
        V_old = [row[:] for row in V]
        for r in range(grid.rows):
            for c in range(grid.cols):
                cell = (r, c)
                if cell in grid.walls or cell in grid.terminals:
                    continue
                new_value = float('-inf')
                for action in grid.actions:
                    next_cell, reward = grid.step(cell, action)
                    new_value = max(
                        new_value, reward + gamma * V_old[next_cell[0]][next_cell[1]]
                    )
                V[r][c] = new_value
                delta = max(delta, abs(V[r][c] - V_old[r][c]))
        show_V(grid, V)
    return V

In [640]:
V = value_iteration(default_grid)

 0.0  0.0  1.0    0 
 0.0    0  0.0  1.0 
 0.0  0.0  0.0  0.0 
------------------
 0.0  0.9  1.0    0 
 0.0    0  0.9  1.0 
 0.0  0.0  0.0  0.9 
------------------
0.81  0.9  1.0    0 
 0.0    0  0.9  1.0 
 0.0  0.0 0.81  0.9 
------------------
0.81  0.9  1.0    0 
0.73    0  0.9  1.0 
 0.0 0.73 0.81  0.9 
------------------
0.81  0.9  1.0    0 
0.73    0  0.9  1.0 
0.66 0.73 0.81  0.9 
------------------
0.81  0.9  1.0    0 
0.73    0  0.9  1.0 
0.66 0.73 0.81  0.9 
------------------


In [641]:
def read_policy(grid: Grid, V, gamma = 0.9):
    policy = [[None for _ in range(grid.cols)] for _ in range(grid.rows)]
    for r in range(grid.rows):
        for c in range(grid.cols):
            cell = (r, c)
            if cell in grid.walls or cell in grid.terminals:
                continue

            max_value = float('-inf')
            for action in grid.actions:
                next_cell, reward = grid.step(cell, action)
                action_value = reward + gamma * V[next_cell[0]][next_cell[1]]
                if action_value > max_value:
                    policy[r][c] = action
                    max_value= action_value
    return policy

policy = read_policy(default_grid, V)
policy

[['right', 'right', 'right', None],
 ['up', None, 'up', 'up'],
 ['up', 'right', 'up', 'up']]

In [642]:
def render_policy(grid: Grid, policy):
    for r in range(len(policy)):
        for c in range(len(policy[0])):
            if policy[r][c] != None:
                value = grid.arrows[policy[r][c]]
            else:
                value = grid.cell_repr(r, c)
            print(f' {value} ', end='')
        print()
    print('------------------')
render_policy(default_grid, policy)

 →  →  →  1 
 ↑  #  ↑  ↑ 
 ↑  →  ↑  ↑ 
------------------


In [643]:
def policy_evaluation(grid: Grid, policy, gamma = 0.9, verbose = False):
    V = [[0 for _ in range(grid.cols)] for _ in range(grid.rows)]
    delta = 1e9
    while delta > 0.001:
        delta = 0
        V_old = [row[:] for row in V]
        for r in range(grid.rows):
            for c in range(grid.cols):
                cell = (r, c)
                if cell in grid.walls or cell in grid.terminals:
                    continue
                action = policy[r][c]
                next_cell, reward = grid.step(cell, action)
                V[r][c] = reward + gamma * V_old[next_cell[0]][next_cell[1]]
                delta = max(delta, abs(V[r][c] - V_old[r][c]))
        if verbose:
            show_V(grid, V)
    return V

policy_evaluation(default_grid, policy, verbose=True);

 0.0  0.0  1.0    0 
 0.0    0  0.0  1.0 
 0.0  0.0  0.0  0.0 
------------------
 0.0  0.9  1.0    0 
 0.0    0  0.9  1.0 
 0.0  0.0  0.0  0.9 
------------------
0.81  0.9  1.0    0 
 0.0    0  0.9  1.0 
 0.0  0.0 0.81  0.9 
------------------
0.81  0.9  1.0    0 
0.73    0  0.9  1.0 
 0.0 0.73 0.81  0.9 
------------------
0.81  0.9  1.0    0 
0.73    0  0.9  1.0 
0.66 0.73 0.81  0.9 
------------------
0.81  0.9  1.0    0 
0.73    0  0.9  1.0 
0.66 0.73 0.81  0.9 
------------------


In [644]:
def policy_iteration(grid: Grid):
    policy = [['up' for _ in range(grid.cols)] for _ in range(grid.rows)]
    while True:
        V = policy_evaluation(grid, policy)
        show_V(grid, V)
        new_policy = read_policy(grid, V)
        changed = sum(
            new_policy[r][c] != None and new_policy[r][c] != policy[r][c]
            for r in range(grid.rows)
            for c in range(grid.cols)
        )
        print(f'actions changed = {changed}')
        render_policy(grid, new_policy)
        if new_policy == policy:
            break
        policy = new_policy
    return policy, V

In [645]:
policy_iteration(default_grid);

 0.0  0.0  0.0    0 
 0.0    0  0.0  1.0 
 0.0  0.0  0.0  0.9 
------------------
actions changed = 3
 ↑  ↑  →  1 
 ↑  #  →  ↑ 
 ↑  ↑  →  ↑ 
------------------
 0.0  0.0  1.0    0 
 0.0    0  0.9  1.0 
 0.0  0.0 0.81  0.9 
------------------
actions changed = 4
 ↑  →  →  1 
 ↑  #  ↑  ↑ 
 ↑  →  ↑  ↑ 
------------------
 0.0  0.9  1.0    0 
 0.0    0  0.9  1.0 
 0.0 0.73 0.81  0.9 
------------------
actions changed = 2
 →  →  →  1 
 ↑  #  ↑  ↑ 
 →  →  ↑  ↑ 
------------------
0.81  0.9  1.0    0 
0.73    0  0.9  1.0 
0.66 0.73 0.81  0.9 
------------------
actions changed = 1
 →  →  →  1 
 ↑  #  ↑  ↑ 
 ↑  →  ↑  ↑ 
------------------
0.81  0.9  1.0    0 
0.73    0  0.9  1.0 
0.66 0.73 0.81  0.9 
------------------
actions changed = 0
 →  →  →  1 
 ↑  #  ↑  ↑ 
 ↑  →  ↑  ↑ 
------------------
